# 1.00 Flujo completo con el paquete

Con el código refactorizado en `experimentacion_mlflow_abandono_clientes/` esta notebook
solo manda llamar las funciones. El registro del experimento en MLflow se hace desde
terminal con `make train`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import mlflow
import pandas as pd
from sklearn.metrics import classification_report

from experimentacion_mlflow_abandono_clientes.config import TRACKING_URI
from experimentacion_mlflow_abandono_clientes.dataset import cargar_datos, limpiar_datos
from experimentacion_mlflow_abandono_clientes.features import dividir_datos
from experimentacion_mlflow_abandono_clientes.modeling.predict import evaluar_umbral
from experimentacion_mlflow_abandono_clientes.modeling.train import entrenar, evaluar


## Carga y limpieza

In [ ]:
churn_df = limpiar_datos(cargar_datos())
print("Registros finales:", churn_df.shape[0], "| Variables:", churn_df.shape[1])
churn_df.head()

## División de datos

In [ ]:
X_train, X_test, y_train, y_test = dividir_datos(churn_df)

print("Entrenamiento:", X_train.shape, "| % Churn:", round(y_train.mean() * 100, 1))
print("Prueba:", X_test.shape, "| % Churn:", round(y_test.mean() * 100, 1))

## Entrenamiento del modelo base

In [ ]:
modelo, dummy = entrenar(X_train, y_train)

print("Número de variables después del one-hot encoding:",
      len(modelo.named_steps['preprocesamiento'].get_feature_names_out()))

## Evaluación

In [ ]:
metricas_dummy = evaluar(dummy, X_test, y_test)
metricas_train = evaluar(modelo, X_train, y_train)
metricas_test = evaluar(modelo, X_test, y_test)

pd.DataFrame({'Dummy (prueba)': metricas_dummy,
              'LogReg (entrenamiento)': metricas_train,
              'LogReg (prueba)': metricas_test}).T.round(4)

In [ ]:
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred, target_names=['No churn', 'Churn'], digits=3))

In [ ]:
#Efecto de cambiar el umbral de decisión
evaluar_umbral(y_test, y_proba)

## Registro en MLflow

El run completo (métricas, gráficas y modelo) queda registrado al correr `make train`
con el server levantado. Aquí solo se consultan los runs que ya existen:

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
try:
    runs = mlflow.search_runs(experiment_names=['Telco_Churn_Baseline'])
    display(runs[['run_id', 'tags.mlflow.runName', 'status', 'metrics.test_roc_auc']].head())
except Exception as e:
    print("No se pudo consultar mlflow, ¿está corriendo el server?", e)